# 학습 목표 요약
- 작은 파일 문제 이해 – 작은 파일이 성능과 스토리지에 미치는 영향 알기
- 파일 재작성 학습 – rewrite_data_files 명령으로 작은 파일을 병합하고 최적화하는 방법 배우기
- 옵션 활용 – 정렬(sorting), 최소 파일 개수(min-input-files) 조건 등 파라미터 적용법 익히기
- 메타데이터 활용 – Iceberg 메타데이터(.files, manifest, stats)를 통한 성능 최적화 원리 이해

---

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

---

In [2]:
spark = SparkSession.builder \
    .appName("IcebergSparkWithMinIO") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "rest") \
    .config("spark.sql.catalog.my_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.my_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.my_catalog.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.my_catalog.s3.path-style-access", "true") \
    .getOrCreate()

25/10/11 10:00:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS my_catalog.my_database")

DataFrame[]

In [4]:
# NoSuchBucketException 발생 시 MinIO 에서 버킷 생성 후 다시 시도
spark.sql("""
    CREATE TABLE IF NOT EXISTS my_catalog.my_database.my_table_rewrite (
        id BIGINT,
        name STRING,
        age INT
    ) USING iceberg
    TBLPROPERTIES ('format-version'='2');
""")
spark.sql("SHOW TABLES IN my_catalog.my_database").show()

+-----------+----------------+-----------+
|  namespace|       tableName|isTemporary|
+-----------+----------------+-----------+
|my_database|my_table_rewrite|      false|
+-----------+----------------+-----------+



In [5]:
# 현재 Spark 병렬처리 설정값 조회
spark.sparkContext.defaultParallelism

10

In [6]:
spark.sql("""
    INSERT INTO my_catalog.my_database.my_table_rewrite VALUES 
    (1, 'Alice', 25),
    (2, 'Bob', 30),
    (3, 'Charlie', 22);
""")

DataFrame[]

- `spark.sparkContext.defaultParallelism` 가 10으로 설정 되어 있어서 각 행이 3개의 파일로 저장되었다.
- 그리고 file size 는 대략 900바이트씩 된다. 이는 데이터를 조회할 때 여러개의 파일을 스캔해야하기 때문에 최적화된 결과가 아니다.
- Iceberg 을 사용하다보면 이렇게 작은 파일이 여러개 적재 되는 경우가 꼭 발생한다.

In [7]:
spark.sql("SELECT * FROM my_catalog.my_database.my_table_rewrite").show()
spark.sql("""
    SELECT 
        record_count, 
        file_size_in_bytes, 
        file_path  
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 22|
+---+-------+---+

+------------+------------------+-------------------------------------------------------------------------------------------------------------+
|record_count|file_size_in_bytes|file_path                                                                                                    |
+------------+------------------+-------------------------------------------------------------------------------------------------------------+
|1           |922               |s3://warehouse/my_database/my_table_rewrite/data/00000-0-12485713-9f83-471f-9038-0a2413c6ad9a-0-00001.parquet|
|1           |908               |s3://warehouse/my_database/my_table_rewrite/data/00001-1-12485713-9f83-471f-9038-0a2413c6ad9a-0-00001.parquet|
|1           |936               |s3://warehouse/my_database/my_table_rewrite/data/00002-2-12485713-9f83-471f-9038-0a2413c6ad9a-0-00001.parquet|
+--------

- 작은 파일을 더 만들기 위해 3개 행을 추가로 INSERT 한다.

In [8]:
spark.sql("""
    INSERT INTO my_catalog.my_database.my_table_rewrite VALUES 
    (4, 'Jaemin', 23),
    (5, 'Jimmy', 32),
    (6, 'Keo', 24);
""")

DataFrame[]

- 아래 결과와 같이 6개 행이 각각 데이터 파일로 기록 되어 있다.

In [10]:
spark.sql("SELECT * FROM my_catalog.my_database.my_table_rewrite").show()
spark.sql("""
    SELECT 
        record_count, 
        file_size_in_bytes, 
        file_path  
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 22|
|  4| Jaemin| 23|
|  5|  Jimmy| 32|
|  6|    Keo| 24|
+---+-------+---+

+------------+------------------+-------------------------------------------------------------------------------------------------------------+
|record_count|file_size_in_bytes|file_path                                                                                                    |
+------------+------------------+-------------------------------------------------------------------------------------------------------------+
|1           |928               |s3://warehouse/my_database/my_table_rewrite/data/00000-5-68941daa-cbc6-4a49-94ce-59dce88a75c5-0-00001.parquet|
|1           |921               |s3://warehouse/my_database/my_table_rewrite/data/00001-6-68941daa-cbc6-4a49-94ce-59dce88a75c5-0-00001.parquet|
|1           |908               |s3://warehouse/my_database/my_table_rewrite/data/00002-7-68941daa-

## Iceberg `rewrite_data_files` procedure
- 개념설명: https://iceberg.apache.org/docs/1.5.1/spark-procedures/#rewrite_data_files
- 쿼리예제: https://iceberg.apache.org/docs/1.5.1/spark-procedures/#examples_5

In [11]:
spark.sql("""
    CALL my_catalog.system.rewrite_data_files('my_database.my_table_rewrite')
""").show()

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                         6|                     1|                 5523|                      0|
+--------------------------+----------------------+---------------------+-----------------------+



In [12]:
# Spark 캐시를 버리고 Iceberg 카탈로그 최신 정보를 직접 동기화
spark.sql("REFRESH TABLE my_catalog.my_database.my_table_rewrite")

DataFrame[]

- Data file rewrite 후 아래 출력 결과와 같이 각 행별 데이터 파일로 존재하던 parquet 파일이 1개의 데이터 파일로 병합 되었다.
- 그리고 900바이트씩 하던 parquet 파일들이 1개로 병합 되면서 1000바이트 정도로 압축면에도 최적화 되었다.
- 병합 과정에서 정렬 기준이 없었기 때문에 지금은 id 기준으로 정렬 되어 있지 않다.

In [14]:
spark.sql("SELECT * FROM my_catalog.my_database.my_table_rewrite").show()
spark.sql("""
    SELECT 
        record_count, 
        file_size_in_bytes, 
        file_path  
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  4| Jaemin| 23|
|  5|  Jimmy| 32|
|  6|    Keo| 24|
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 22|
+---+-------+---+

+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|record_count|file_size_in_bytes|file_path                                                                                                     |
+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|6           |1028              |s3://warehouse/my_database/my_table_rewrite/data/00000-14-9848fb6b-d070-46af-b782-e99a3e0f978a-0-00001.parquet|
+------------+------------------+--------------------------------------------------------------------------------------------------------------+



### Iceberg `rewrite_data_files` procedure with OPTIONS
- 옵션설명: https://iceberg.apache.org/docs/1.5.1/spark-procedures/#options
- `rewrite_data_files` 프로시저 옵션을 추가하여 테스트하기 위해 작은 파일을 더 만들어보자.

In [15]:
spark.sql("""
    INSERT INTO my_catalog.my_database.my_table_rewrite VALUES 
    (7, 'Boo', 23),
    (8, 'Foo', 32),
    (9, 'Choo', 24);
""")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


DataFrame[]

In [16]:
spark.sql("SELECT * FROM my_catalog.my_database.my_table_rewrite").show()
spark.sql("""
    SELECT 
        record_count, 
        file_size_in_bytes, 
        file_path  
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  4| Jaemin| 23|
|  5|  Jimmy| 32|
|  6|    Keo| 24|
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 22|
|  7|    Boo| 23|
|  8|    Foo| 32|
|  9|   Choo| 24|
+---+-------+---+

+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|record_count|file_size_in_bytes|file_path                                                                                                     |
+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|1           |908               |s3://warehouse/my_database/my_table_rewrite/data/00000-19-94809a17-b9f8-4fbf-8301-9a027ff5379d-0-00001.parquet|
|1           |908               |s3://warehouse/my_database/my_table_rewrite/data/00001-20-94809a17-b9f8-4fbf-8301-9a027ff5379d-0-00001.parquet|
|1           |914               |s3://wa

### SORT, OPTIONS 사용하기
- data file rewrite 시 id 컬럼 기준으로 ASC 정렬한다.
- 각 디렉토리 내 data file 이 최소 5개 이상일 경우에만 rewrite 되도록 한다.

In [17]:
# 현재까지 최신 메타데이터에 활성화 된 데이터 파일 개수는 4개 이므로 rewrite 되지 않아야한다.
spark.sql("""
    CALL my_catalog.system.rewrite_data_files(
        table => 'my_database.my_table_rewrite',
        strategy => 'sort', 
        sort_order => 'id ASC',
        options => map('min-input-files','5')
    )
""").show()

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                         0|                     0|                    0|                      0|
+--------------------------+----------------------+---------------------+-----------------------+



- 위 결과와 같이 데이터 파일이 최소 5개 이상 되어야하는데 4개뿐이라 rewrite 되지 않았다.
- 데이터 파일이 많이 존재하지 않는데 매번 rewrite 하는 것은 매우 큰 낭비일 수 있으니 실제 사용 환경에서도 주의해야한다.
- 행을 더 삽입하여 다시 rewrite 시도해보자.

In [18]:
spark.sql("""
    INSERT INTO my_catalog.my_database.my_table_rewrite VALUES 
    (10, 'Yoo', 23),
    (11, 'Hee', 32),
    (12, 'Coo', 24);
""")

DataFrame[]

In [19]:
spark.sql("SELECT * FROM my_catalog.my_database.my_table_rewrite").show()
spark.sql("""
    SELECT 
        record_count, 
        file_size_in_bytes, 
        file_path  
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  7|    Boo| 23|
|  8|    Foo| 32|
|  9|   Choo| 24|
|  4| Jaemin| 23|
|  5|  Jimmy| 32|
|  6|    Keo| 24|
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 22|
| 10|    Yoo| 23|
| 11|    Hee| 32|
| 12|    Coo| 24|
+---+-------+---+

+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|record_count|file_size_in_bytes|file_path                                                                                                     |
+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|1           |908               |s3://warehouse/my_database/my_table_rewrite/data/00000-24-658cc749-01d8-40e0-96b9-05e7aa1cd888-0-00001.parquet|
|1           |908               |s3://warehouse/my_database/my_table_rewrite/data/00001-25-658cc749-01d8-40e0-96b9-05e7aa1cd888-0-0

In [20]:
# 현재까지 최신 메타데이터에 활성화 된 데이터 파일 개수는 7개 이므로 rewrite 되어야한다.
spark.sql("""
    CALL my_catalog.system.rewrite_data_files(
        table => 'my_database.my_table_rewrite',
        strategy => 'sort', 
        sort_order => 'id ASC',
        options => map('min-input-files','5')
    )
""").show()

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                         7|                     1|                 6481|                      0|
+--------------------------+----------------------+---------------------+-----------------------+



- 7개의 데이터 파일이 1개로 병합 되었다.
- 아래와 같이 SELECT 쿼리 시 ORDER BY 를 지정하지 않아도 id 컬럼으로 ASC 정렬 되어있다.(실제 parquet 내부적으로 id 컬럼으로 정렬 되어 있다.)

In [22]:
spark.sql("SELECT * FROM my_catalog.my_database.my_table_rewrite").show()
spark.sql("""
    SELECT 
        record_count, 
        file_size_in_bytes, 
        file_path  
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 25|
|  2|    Bob| 30|
|  3|Charlie| 22|
|  4| Jaemin| 23|
|  5|  Jimmy| 32|
|  6|    Keo| 24|
|  7|    Boo| 23|
|  8|    Foo| 32|
|  9|   Choo| 24|
| 10|    Yoo| 23|
| 11|    Hee| 32|
| 12|    Coo| 24|
+---+-------+---+

+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|record_count|file_size_in_bytes|file_path                                                                                                     |
+------------+------------------+--------------------------------------------------------------------------------------------------------------+
|12          |1116              |s3://warehouse/my_database/my_table_rewrite/data/00000-31-f0399490-c52d-4702-9910-31a879610c2c-0-00001.parquet|
+------------+------------------+--------------------------------------------------------------------------------------------------

---

# Table files view 확인
- 앞서 사용한 files view 를 조금 더 살펴보자.

## NULL Count 확인
- 병합된 파일에서 null 값 여부를 확인해보자.
- `{1 -> 0, 2 -> 0, 3 -> 0}` 일 경우 1번 컬럼의 null 은 0건, 2번 컬럼의 null은 0건, 3번 컬럼의 null은 0건 이라는 의미이다.

In [23]:
spark.sql("""
    SELECT 
        null_value_counts
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+------------------------+
|null_value_counts       |
+------------------------+
|{1 -> 0, 2 -> 0, 3 -> 0}|
+------------------------+



## 상한값, 하한값 확인해보기
- Iceberg 메타데이터에는 각 컬럼별 상한, 하한값을 집계해둔 lower_bounds, uppger_bounds 정보가 있다.
- 각 정보는 16진수 문자열로 기록 되며, 
- lower_bounds: `{1 -> [01 00 00 00 00 00 00 00], 2 -> [41 6C 69 63 65], 3 -> [16 00 00 00]}`와 같을 경우 1번 컬럼의 하한값은 [01 00 00 00 00 00 00 00] 라는 의미이다.

In [27]:
spark.sql("""
    SELECT 
        lower_bounds, 
        upper_bounds
    FROM my_catalog.my_database.my_table_rewrite.files
""").show(truncate=False)

+---------------------------------------------------------------------------+---------------------------------------------------------------------+
|lower_bounds                                                               |upper_bounds                                                         |
+---------------------------------------------------------------------------+---------------------------------------------------------------------+
|{1 -> [01 00 00 00 00 00 00 00], 2 -> [41 6C 69 63 65], 3 -> [16 00 00 00]}|{1 -> [0C 00 00 00 00 00 00 00], 2 -> [59 6F 6F], 3 -> [20 00 00 00]}|
+---------------------------------------------------------------------------+---------------------------------------------------------------------+



In [48]:
# 데이터파일 1번 컬럼의 하한값
byte_data = bytes.fromhex("01 00 00 00 00 00 00 00")
int.from_bytes(byte_data, byteorder='little', signed=True)

1

In [45]:
# 데이터파일 2번 컬럼의 하한값
bytes.fromhex("41 6C 69 63 65").decode("utf-8")

'Alice'

In [47]:
# 데이터파일 3번 컬럼의 하한값
byte_data = bytes.fromhex("16 00 00 00")
int.from_bytes(byte_data, byteorder='little', signed=True)

22

In [49]:
# 데이터파일 1번 컬럼의 상한값
byte_data = bytes.fromhex("0C 00 00 00 00 00 00 00")
int.from_bytes(byte_data, byteorder='little', signed=True)

12

In [50]:
# 데이터파일 2번 컬럼의 상한값
bytes.fromhex("59 6F 6F").decode("utf-8")

'Yoo'

In [51]:
# 데이터파일 3번 컬럼의 상한값
byte_data = bytes.fromhex("20 00 00 00")
int.from_bytes(byte_data, byteorder='little', signed=True)

32

---

# Spark 병렬성 조정 방법
- 앞서 `spark.sparkContext.defaultParallelism` 가 10인 걸 확인했고, 병렬성에 맞춰 parquet 파일이 개별로 적재 되었다.
- spark config 설정에서 병렬성을 조정할 수 있다.

In [53]:
# 현재 defaultParallelism
spark.sparkContext.defaultParallelism

10

- 왜 10일까?
- 아래와 같이 로컬에서 실행하는 모든 자원([*])을 사용하도록 되어 있는데 내 작업 환경에서 spark-iceberg 컨테이너에 할당된 코어 수가 10개이기 때문이다.
- 나는 `docker exec -it spark-iceberg nproc` 로 조회하면 10이 나온다.

In [54]:
spark.sparkContext.master

'local[*]'

- 아래와 같이 spark.stop 후 SparkSession 빌드 시 `spark.sparkContext.defaultParallelism` 값을 원하는 숫자로 변경하여 실행하면 된다.

In [55]:
spark.stop()
spark = SparkSession.builder \
            .config("spark.default.parallelism", "3") \
            .getOrCreate()
spark.sparkContext.defaultParallelism

25/10/11 10:30:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3

- 실제 환경에서는 Spark App 별로 spark.default.parallelism 값을 조정하거나 DataFrame 으로 작업할 경우 파티션 수를 조정하여 대응할 수 있다.